# Healthcare Fraud Detection — Validation-Selected Thresholds

**New post-filing methodological improvement (November 2026).** This notebook uses a stratified provider-level train/validation/test split. It selects decision thresholds on the validation set only, then reports untouched-test performance. It does not establish temporal generalization, external validity, claim-level prepayment performance, or deployment. Results are benchmark-only.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, json
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve
ROOT=Path.cwd(); OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(OUT/'provider_level_features.csv')
features=[c for c in df if c not in ['Provider','PotentialFraud','Label']]
X=df[features]; y=df.Label
X_dev,X_test,y_dev,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_dev,y_dev,test_size=.25,random_state=43,stratify=y_dev)
print('Train/validation/test providers:',len(X_train),len(X_val),len(X_test))
models={
 'Logistic regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1500,random_state=42)),
 'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingClassifier(random_state=42))
}
def choose_threshold(y_true,prob,target=.70):
 fpr,tpr,thresholds=roc_curve(y_true,prob)
 valid=np.flatnonzero(tpr>=target)
 idx=valid[np.argmin(fpr[valid])]
 return float(thresholds[idx])
rows=[]; export=[]
for name,model in models.items():
 model.fit(X_train,y_train)
 pv=model.predict_proba(X_val)[:,1]
 threshold=choose_threshold(y_val,pv,.70)
 pt=model.predict_proba(X_test)[:,1]
 tn,fp,fn,tp=confusion_matrix(y_test,pt>=threshold,labels=[0,1]).ravel()
 rows.append({'model':name,'validation_selected_threshold':threshold,'test_roc_auc':roc_auc_score(y_test,pt),'test_average_precision':average_precision_score(y_test,pt),'test_recall':tp/(tp+fn),'test_fpr':fp/(fp+tn),'test_precision':tp/(tp+fp) if tp+fp else 0,'test_tp':int(tp),'test_fp':int(fp),'test_tn':int(tn),'test_fn':int(fn)})
 export.extend([{'model':name,'score':float(v),'label':int(label)} for v,label in zip(pt,y_test)])
metrics=pd.DataFrame(rows); print(metrics.to_string(index=False))
metrics.to_csv(OUT/'powerbi_corrected_model_metrics.csv',index=False)
pd.DataFrame(export).to_csv(OUT/'corrected_test_scores.csv',index=False)
print('Saved corrected validation metrics and test scores.')

Train/validation/test providers: 3246 1082 1082


              model  validation_selected_threshold  test_roc_auc  test_average_precision  test_recall  test_fpr  test_precision  test_tp  test_fp  test_tn  test_fn
Logistic regression                       0.659528      0.959266                0.748910     0.811881  0.083588        0.500000       82       82      899       19
  Gradient boosting                       0.224769      0.957045                0.692165     0.772277  0.064220        0.553191       78       63      918       23
Saved corrected validation metrics and test scores.


### Interpretation

Threshold selection is now separate from the test set. The benchmark is still retrospective and uses provider-level labels. Comparisons between the original and revised metrics are not strictly apples-to-apples because the split and training sizes differ. External and temporal validation remain outstanding.